<a href="https://colab.research.google.com/github/Yipweiyang/SIT-UofG-QC-Assignment/blob/main/BB84-Attacker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 87.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 5.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=eec0b882824e337f1e09b1d5f57930c7d2a33d260fbe403b31ba9da14c71abf2
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [ ]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

In [2]:
# Quantum random bit generator

def quantum_random_bit():
    qc = QuantumCircuit(1, 1)

    qc.h(0)

    qc.measure(0, 0)

    simulator = BasicSimulator()
    result = simulator.run(qc, shots=1).result()

    counts = result.get_counts()

    measured_bit = list(counts.keys())[0]

    return int(measured_bit)

In [3]:
# Alice generates random bits and random bases

n = 40  # use more qubits so attack detection is clearer

alice_bits = []
alice_bases = []

for i in range(n):
    alice_bits.append(quantum_random_bit())

    if quantum_random_bit() == 0:
        alice_bases.append("s")
    else:
        alice_bases.append("d")

print("Alice bits: ", alice_bits)
print("Alice bases:", alice_bases)

Alice bits:  [1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1]
Alice bases: ['d', 's', 'd', 's', 'd', 'd', 's', 'd', 's', 's', 's', 'd', 'd', 'd', 'd', 'd', 's', 'd', 'd', 'd', 's', 'd', 'd', 'd', 's', 'd', 's', 's', 's', 'd', 'd', 'd', 'd', 'd', 'd', 's', 's', 's', 's', 's']


In [4]:
# Alice prepares qubits

alice_qubits = []

for i in range(n):

    qc = QuantumCircuit(1, 1)

    bit = alice_bits[i]
    basis = alice_bases[i]

    # Encode the bit
    if bit == 1:
        qc.x(0)

    # Encode the basis
    if basis == "d":
        qc.h(0)

    alice_qubits.append(qc)

print("Alice prepared", len(alice_qubits), "qubits.")

Alice prepared 40 qubits.


In [5]:
# Eve intercepts Alice's qubits and resends new qubits to Bob

eve_bases = []
eve_bits = []
eve_qubits = []

simulator = BasicSimulator()

for i in range(n):

    # Eve randomly chooses a measurement basis
    if quantum_random_bit() == 0:
        eve_basis = "s"
    else:
        eve_basis = "d"

    eve_bases.append(eve_basis)

    # Eve copies Alice's qubit circuit so she can measure it
    qc = alice_qubits[i].copy()

    # If Eve uses diagonal basis, apply H before measuring
    if eve_basis == "d":
        qc.h(0)

    qc.measure(0, 0)

    result = simulator.run(qc, shots=1).result()
    counts = result.get_counts()

    eve_bit = int(list(counts.keys())[0])
    eve_bits.append(eve_bit)

    # Eve prepares a new qubit based on what she measured
    resend_qc = QuantumCircuit(1, 1)

    if eve_bit == 1:
        resend_qc.x(0)

    if eve_basis == "d":
        resend_qc.h(0)

    eve_qubits.append(resend_qc)

print("Eve bases:", eve_bases)
print("Eve bits: ", eve_bits)
print("Eve resent", len(eve_qubits), "qubits to Bob.")

Eve bases: ['d', 's', 'd', 'd', 's', 's', 'd', 'd', 's', 'd', 'd', 'd', 'd', 's', 'd', 'd', 'd', 'd', 's', 's', 's', 'd', 's', 's', 's', 's', 'd', 'd', 'd', 'd', 's', 'd', 's', 's', 's', 'd', 's', 's', 's', 's']
Eve bits:  [1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1]
Eve resent 40 qubits to Bob.
